# OpenAI Function Calling Demo

This notebook demonstrates how to use OpenAI's Function Calling feature (Tools) to connect an LLM to external data sources and APIs.

We will implement three tools:
1.  **`get_weather`**: Fetches weather data from the Open-Meteo API.
2.  **`get_flight`**: Queries a local DuckDB database for flight information.
3.  **`get_fact`**: Queries a local DuckDB database for fun facts about a city.

In [ ]:
!uv pip install -q openai duckdb requests python-dotenv

In [ ]:
import os
import json
import requests
import duckdb
from openai import OpenAI

# --- API Key Setup ---
# Option 1 — Google Colab: Load API key from Colab Secrets
# from google.colab import userdata
# api_key = userdata.get('OPENAI_API_KEY')

# Option 2 — Local (VSCode / Jupyter): Load API key from .env file
# from dotenv import load_dotenv
# load_dotenv()
# api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("⚠️  OPENAI_API_KEY not found. Set it in Colab Secrets or a .env file.")
else:
    print("✅ API key loaded.")

client = OpenAI(api_key=api_key)

## 1. Setup Database

We will use DuckDB to create an in-memory database and load our CSV files.

In [ ]:
# Connect to DuckDB (using a file-based db or in-memory)
# Using a file-based DB 'city_tour.db' as per the user request, or we could use ':memory:'
conn = duckdb.connect("city_tour.db")

# Load datasets from CSV files into the database
# We use CREATE OR REPLACE TABLE to avoid errors if run multiple times
try:
    conn.execute("""
        CREATE OR REPLACE TABLE flight AS 
        SELECT * FROM 'flight_data.csv'
    """)
    print("Flight table created.")
    
    conn.execute("""
        CREATE OR REPLACE TABLE fun_facts AS 
        SELECT * FROM 'fun_facts.csv'
    """)
    print("Fun Facts table created.")
    
except Exception as e:
    print(f"Error loading data: {e}")

# Verify tables
print("Tables:", conn.execute("SHOW TABLES").fetchall())

## 2. Define Python Functions

These are the actual functions that will be executed when the model requests them.

In [ ]:
def get_weather(latitude, longitude):
    """Get the current temperature for a given latitude and longitude."""
    print(f"DEBUG: Calling get_weather for {latitude}, {longitude}")
    try:
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m,wind_speed_10m",
            "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m"
        }
        response = requests.get(url, params=params)
        data = response.json()
        return json.dumps(data['current'])
    except Exception as e:
        return json.dumps({"error": str(e)})

def get_flight(from_city, to_city):
    """Get flight information between two cities."""
    print(f"DEBUG: Calling get_flight from {from_city} to {to_city}")
    try:
        # Use parameterized queries to prevent injection (though low risk here)
        results = conn.execute(
            "SELECT * FROM flight WHERE from_city=? AND to_city=?", 
            [from_city, to_city]
        ).fetchall()
        
        if not results:
            return json.dumps({"error": "No flights found"})
        
        # Convert list of tuples to list of dicts or string for better readability by LLM
        columns = [desc[0] for desc in conn.description]
        formatted_results = [
            dict(zip(columns, row)) for row in results
        ]
        return json.dumps(formatted_results)
    except Exception as e:
        return json.dumps({"error": str(e)})

def get_fact(city):
    """Get a fun fact about a specific city."""
    print(f"DEBUG: Calling get_fact for {city}")
    try:
        results = conn.execute(
            "SELECT * FROM fun_facts WHERE City=?", 
            [city]
        ).fetchall()
        
        if not results:
            return json.dumps({"error": "No facts found"})
        
        columns = [desc[0] for desc in conn.description]
        formatted_results = [
            dict(zip(columns, row)) for row in results
        ]
        return json.dumps(formatted_results)
    except Exception as e:
        return json.dumps({"error": str(e)})

## 3. Define Tools Schema

This is the metadata that we send to OpenAI so it knows what tools are available and how to call them.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {
                        "type": "number",
                        "description": "The latitude of the city"
                    },
                    "longitude": {
                        "type": "number",
                        "description": "The longitude of the city"
                    }
                },
                "required": ["latitude", "longitude"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_flight",
            "description": "Get flight information between two cities",
            "parameters": {
                "type": "object",
                "properties": {
                    "from_city": {
                        "type": "string",
                        "description": "The departure city"
                    },
                    "to_city": {
                        "type": "string",
                        "description": "The destination city"
                    }
                },
                "required": ["from_city", "to_city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_fact",
            "description": "Get a fun fact about a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

## 4. Run Conversation with Tools

This is the core logic that orchestrates the conversation. The **Function Calling** lifecycle consists of 4 main steps:

1.  **Initial Call**: We send the user's prompt to the model along with the `tools` definitions. The model does *not* execute the code itself; instead, it decides if it *needs* to call a function to answer the question.
2.  **Check for Tool Calls**: We inspect the model's response (`response_message.tool_calls`).
    *   If **No Tool Needed**: The model returns a normal text response (e.g., "Hello!"), and we are done.
    *   If **Tool Needed**: The model returns a "call request" containing the function name (e.g., `get_weather`) and the arguments (e.g., `{"latitude": 25, "longitude": 55}`).
3.  **Execute Tool**: We (the code) catch this request, find the corresponding Python function, run it, and capture the output (usually a JSON string).
4.  **Submit Result**: We add the tool's output to the conversation history as a new message with role `tool` and send the entire history back to the model. The model now uses this data to generate the final natural language answer.

In [ ]:
def run_conversation(prompt):
    # Step 1: Send the conversation and available tools to the model
    messages = [{"role": "user", "content": prompt}]
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto",  # auto is default, but we declare it explicitly
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls
    
    # Step 2: Check if the model wanted to call a function
    if tool_calls:
        print("Model requested tools:", len(tool_calls))
        
        available_functions = {
            "get_weather": get_weather,
            "get_flight": get_flight,
            "get_fact": get_fact,
        }
        
        # Add the assistant's response (with tool_calls) to the conversation
        messages.append(response_message)
        
        # Step 3: Execute each requested tool call
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_to_call = available_functions[function_name]
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"Executing {function_name} with args: {function_args}")
            
            if function_name == "get_weather":
                function_response = function_to_call(
                    latitude=function_args.get("latitude"),
                    longitude=function_args.get("longitude"),
                )
            elif function_name == "get_flight":
                function_response = function_to_call(
                    from_city=function_args.get("from_city"),
                    to_city=function_args.get("to_city"),
                )
            elif function_name == "get_fact":
                function_response = function_to_call(
                    city=function_args.get("city"),
                )
            
            # Add the tool result to the conversation with role="tool"
            messages.append(
                {
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": function_response,
                }
            )
        
        # Step 4: Send the updated conversation (with tool results) back to the model
        second_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
        )
        return second_response.choices[0].message.content
        
    else:
        # No tool call needed — return the direct response
        return response_message.content

## 5. Examples

In [ ]:
# Example 1: Weather (Requires Lat/Long)
# Note: The model might ask for a city name first, effectively chaining calls if we had a geocoding tool,
# but here we will provide specific lat/long or the model might have internal knowledge of lat/long for major cities.
# Let's ask specifically with coordinates to test our tool directly.
# Dubai coordinates: 25.2048, 55.2708
print("User: What is the weather like in these coordinates 25.2048, 55.2708?")
print("Assistant:", run_conversation("What is the weather like in these coordinates 25.2048, 55.2708?"))

In [ ]:
# Example 2: Flights
print("User: Is there a flight from Paris to London?")
# Note: Case sensitivity might be an issue depending on how the CSV data is formatted. 
# The function handles it by passing strings directly to SQL.
print("Assistant:", run_conversation("Is there a flight from Paris to London?"))

In [ ]:
# Example 3: Fun Fact
print("User: Tell me a fun fact about Tokyo.")
print("Assistant:", run_conversation("Tell me a fun fact about Tokyo."))

In [ ]:
# Example 4: Complex Query (Flight + Fact)
print("User: Find me a flight from New York to London and tell me a fun fact about London.")
print("Assistant:", run_conversation("Find me a flight from New York to London and tell me a fun fact about London."))